# 🧬 De Novo Immunopeptidomics — CNN-LSTM Training

**Target:** Exact peptide accuracy ≥ 30%  
**GPU required:** Enable via *Runtime → Change runtime type → T4 GPU*

---
### Before you start
1. Click **Runtime → Change runtime type** and select **T4 GPU**.
2. Run **Section 1** to verify you have a GPU.
3. Run **Section 2** to clone the repo.
4. Run **Section 3** to upload your data files.
5. Run **Section 4** to install dependencies.
6. Run **Section 5** to train.
7. Run **Section 6** to download the best checkpoint.

## Section 1 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## Section 2 — Clone the Repository

In [ ]:
import os

REPO_URL = 'https://github.com/chandan-rawallab/immunodenovorepo.git'
REPO_DIR = '/content/immunodenovorepo'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!ls src/

## Section 3 — Upload Data

Upload your files using the cell below **OR** mount Google Drive if your data is there.

Required files:
- `immunopeptidome_psms.tsv` — PSM labels
- `*.mgf` — raw spectra files (one per run)

In [ ]:
# ── Option A: Upload files directly ─────────────────────────────────────────
from google.colab import files

DATA_DIR   = '/content/immunodenovorepo/data'
MGF_DIR    = DATA_DIR + '/mgf'
RESULTS    = '/content/immunodenovorepo/results'

import os
os.makedirs(MGF_DIR, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

print('Upload immunopeptidome_psms.tsv first:')
uploaded = files.upload()
for fname, data in uploaded.items():
    out_path = RESULTS + '/' + fname
    with open(out_path, 'wb') as f:
        f.write(data)
    print(f'  Saved → {out_path}')

print()
print('Now upload your .mgf files (select all at once):')
uploaded_mgf = files.upload()
for fname, data in uploaded_mgf.items():
    out_path = MGF_DIR + '/' + fname
    with open(out_path, 'wb') as f:
        f.write(data)
    print(f'  Saved → {out_path}')

In [ ]:
# ── Option B: Mount Google Drive instead ────────────────────────────────────
# Uncomment and run this cell if your data is already on Drive.

# from google.colab import drive
# drive.mount('/content/drive')
# MGF_DIR  = '/content/drive/MyDrive/hla_data/mgf'    # ← change to your path
# PSM_FILE = '/content/drive/MyDrive/hla_data/immunopeptidome_psms.tsv'

## Section 4 — Install Dependencies

In [ ]:
%%bash
pip install -q pyteomics lxml pandas scikit-learn tqdm
echo 'Dependencies installed.'

## Section 5 — Train

In [ ]:
# ── Configure paths ──────────────────────────────────────────────────────────
# If you used Option A above these defaults are correct.
# If you used Google Drive (Option B) update MGF_DIR and PSM_FILE here.

MGF_DIR       = '/content/immunodenovorepo/data/mgf'
PSM_FILE      = '/content/immunodenovorepo/results/immunopeptidome_psms.tsv'
CHECKPOINT_DIR= '/content/immunodenovorepo/results/checkpoints_v3'

# ── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE     = 32    # fits comfortably on T4 16 GB
EPOCHS         = 150
LR             = 3e-4
LABEL_SMOOTH   = 0.10
PATIENCE       = 25
GRAD_ACCUM     = 2     # effective batch = 64
T0             = 10
T_MULT         = 2

print('Config:')
print(f'  MGF dir      : {MGF_DIR}')
print(f'  PSM file     : {PSM_FILE}')
print(f'  Checkpoint   : {CHECKPOINT_DIR}')
print(f'  Batch size   : {BATCH_SIZE}  (effective {BATCH_SIZE*GRAD_ACCUM})')
print(f'  Epochs       : {EPOCHS}')

In [ ]:
import sys
sys.path.insert(0, '/content/immunodenovorepo/src')

from training.train_denovo_model_05 import train_production

# If the import fails try:
# import importlib.util, pathlib
# spec = importlib.util.spec_from_file_location(
#     'train', '/content/immunodenovorepo/src/training/05_train_denovo_model.py')
# m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
# train_production = m.train_production

train_production(
    mgf_dir=MGF_DIR,
    psm_file=PSM_FILE,
    checkpoint_dir=CHECKPOINT_DIR,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LR,
    num_workers=2,
    t0=T0,
    t_mult=T_MULT,
    label_smoothing=LABEL_SMOOTH,
    patience=PATIENCE,
    grad_accum_steps=GRAD_ACCUM,
)

## Section 6 — Download Checkpoint

In [ ]:
import glob
from google.colab import files

checkpoints = sorted(glob.glob(CHECKPOINT_DIR + '/*.pth'))
if not checkpoints:
    print('No checkpoints found — did training complete?')
else:
    best = [c for c in checkpoints if 'best' in c]
    to_download = best if best else checkpoints[-1:]
    for path in to_download:
        print(f'Downloading {path} ...')
        files.download(path)
    # Also download metadata sidecar
    for path in to_download:
        meta = path.replace('.pth', '_metadata.json')
        if __import__('os').path.exists(meta):
            files.download(meta)

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `ModuleNotFoundError: pyteomics` | Re-run Section 4 |
| `No labelled spectra found` | Check PSM file path; ensure `run_id` values match `.mgf` filenames |
| Runtime disconnects after ~12h | Colab free tier limit. Save checkpoint to Drive: `!cp {CHECKPOINT_DIR}/*.pth /content/drive/MyDrive/` |
| CUDA OOM | Reduce `BATCH_SIZE` to 16 |
| `import train_production` fails | Use the `importlib` fallback in the comment block |